# Connecting to Hardware

**SUMMARY**: *In this lab, you'll learn how to setup and connect to your ChipWhisperer hardware. We'll also cover how to build firmware for your target microcontroller, how to capture power traces, and how to communicate with target devices.*

*All the API calls we'll be using here are documented on the [ChipWhisperer readthedocs page](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html), so feel free to open it in another tab and follow along there as well.*

**LEARNING OUTCOMES:**
* Setting up your ChipWhisperer Hardware
* Using the ChipWhisperer Python API to connect to your hardware
* Communication with the target
* Capturing a power trace

## Prerequisites

Hold up - before continuing, ensure you have done the following:

* ☑ Validated your setup using the [ChipWhisperer Setup Test](./ChipWhisperer%20Setup%20Test.ipynb) notebook.
* ☑ Run through the Jupyter introduction.

## Physical Setup

### CW-Husky with CW313/SAM4S

The only connections you need to make are with the 20-pin connector and the two SMA cables. The only settings to confirm are that JP2 is set to T-GPIO4 and that JP1 and JP4 are connected:

<img src="img/cwhusky.jpg" alt="CW-Husky Plugged In" width=500>

For the tutorials, you should use `PLATFORM='CWHUSKY'`

### ChipWhisperer-Nano

The ChipWhisperer-Nano is a single-board device. It includes the capture hardware, along with a built in STM32F0 target. To use this target, simply plug in the device, then go get a drink to reward yourself:

<img src="img/cwnano.jpg" alt="CW-Nano Plugged In" width=400>

For the tutorials, you should use `PLATFORM='CWNANO'`.

### ChipWhisperer-Lite (Single Board)

The ChipWhisperer-Lite is a single-board device. It includes the capture hardware, along with a built in STM32F3 or XMEGA target. To use this target, simply plug in the device:

<img src="img/cwlite_plugged.jpg" alt="CW-Lite Plugged In" width=400>

If you have an STM32F3 target, you should use `PLATFORM='CWLITEARM'` for the tutorials. If you have an XMEGA target, you should use `PLATFORM='CWLITEXMEGA'`.

### ChipWhisperer-Lite (2-Part Version)

The ChipWhisperer-Lite 2-Part version includes the ChipWhisperer-Capture along with an XMEGA target. You need to attach the SMA & IDE cables, then plug in the USB cable:

<img src="img/cw_2part.jpg" alt="CW-Lite 2-Part Connected" width=500>

For the tutorials, you should use `PLATFORM='CWLITEXMEGA'`.

### ChipWhisperer-Lite (Capture + UFO), Includes SCAPACK-L1/SCAPACK-L2

This package uses a ChipWhisperer-Lite Capture board, along with a CW308 UFO target board. You'll need to attach the CW308 board to your ChipWhisperer-Lite Capture, and plug in your choosen target board. The suggested target for most tutorials is the `CW308_STM32F3` - you can tell this target by checking the part number on the chip, look for a `STM32F303`.

Here is an overview photo of the target plugged in:

<img src="img/cwcapture_ufo.jpg" alt="CW-Lite Plugged Into UFO" width=600>

If using the F3 for the tutorials, you should use `PLATFORM='CW308_STM32F3'`.

See Section below on UFO Target Settings.

### ChipWhisperer-Pro

This package uses a ChipWhisperer-Pro Capture device, along with a CW308 UFO target board. You'll need to attach the CW308 board to your ChipWhisperer-Pro Capture, and plug in your choosen target board. The suggested target for most tutorials is the `CW308_STM32F3` - you can tell this target by checking the part number on the chip, look for a `STM32F303`.

Here is an overview photo of the business end of the CW-Pro plugged into the UFO board. You'll need to also plug in the USB-A cable, and possibly the DC power jack (2.1mm DC power jack, 5V comes from either USB or wall adapter). Earlier hardware revisions always needed the DC power jack, later hardware revisions will use USB power (but high-power targets still need extra juice).

<img src="img/cwpro_ufo.jpg" alt="CW-Pro Plugged In" width=500>

If using the F3 for the tutorials, you should use `PLATFORM='CW308_STM32F3'`.

See Section below on UFO Target Settings.

### UFO Target Settings

The UFO Board comes by default with working settings for most targets. A summary of the default jumper settings is included below for you, see the product documentation for more details.

<img src="img/cwufo_stm32f3.jpg" alt="UFO Board" width=500>

#### UFO Default Settings

* J1  - 3.3V (CW3.3V)
* J3  - HS2/OUT
* J14 - FILT/FILT_LP/FILT_HP
* J16 - No connection

* S1 -  ON

* 3.3V SRC - J1/CW
* VADJ SRC - 3.3V
* 1.2V, 1.8V, 2.5V, 3.3V, LDO SRC - J1/CW
* 5V SRC - J1/CW

### CW-Lite single board: XMEGA vs ARM

If you are unsure which single-part version you have, the following shows both boards. Note the only difference is the target section on the right.

<img src="img/cwlitearm_vs_cwlitexmega.jpg" alt="CW-Lite XMEGA vs Arm Board" width=500>

If you have an STM32F3 target, you should use `PLATFORM='CWLITEARM'` for the tutorials. If you have an XMEGA target, you should use `PLATFORM='CWLITEXMEGA'`.

## Connecting to ChipWhisperer

Now that your hardware is all setup, we can now learn how to connect to it. We can connect to the ChipWhisperer with:

In [1]:
import chipwhisperer as cw
scope = cw.scope()

import time

By default, ChipWhisperer will try to autodetect the type of device your're running (CWLite/CW1200 or CWNano), see API documentation for manually specifying the scope type. If you have multiple ChipWhisperer devices connected, you'll need to specify the serial number of the device you want to connect to:

```python
scope = cw.scope(sn='<some long string of numbers>')
```

For more information, see the [API section on readthedocs](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html).

Connecting to the target device is simple as well:

In [2]:
target = cw.target(scope, cw.targets.SimpleSerial2) #cw.targets.SimpleSerial can be omitted

We'll only be discussing the default target type, which is SimpleSerial. Other targets, like the CW305, will be covered in hardware specific demos. 

Some sane default settings can be set using:

In [3]:
scope.default_setup()

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 327828                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

which from its [documentation](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html#chipwhisperer.scopes.OpenADC.default_setup) you can see does the following for the CWLite/CW1200:

* Sets the scope gain to 45dB
* Sets the scope to capture 5000 samples
* Sets the scope offset to 0 (aka it will begin capturing as soon as it is triggered)
* Sets the scope trigger to rising edge
* Outputs a 7.37MHz clock to the target on HS2
* Clocks the scope ADC at 4\*7.37MHz. Note that this is *synchronous* to the target clock on HS2
* Assigns GPIO1 as serial RX
* Assigns GPIO2 as serial TX

And that's it! Your ChipWhisperer is now setup and ready to attack a target. 

**NOTE: You'll need to disconnect the scope/target before connecting again, like you would in another notebook. This can be done with `scope.dis()` and `target.dis()`**

## Building and Uploading Firmware

The next step in attacking a target is to get some firmware built and uploaded onto it. Most firmware for most ChipWhisperer targets can be built using our build system, provided you have the correct compiler installed (see https://chipwhisperer.readthedocs.io/en/latest/installation.html for info about compilers).

Firmware must be built on the command line. Luckily, thanks to Jupyter, we can run a command within a notebook as follows:

In [4]:
%%bash
cd ../firmware/mcu/simpleserial-base/
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1
#make PLATFORM=CW308_STM32F3 CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Compiling:
-en     simpleserial-base.c ...
-e Done!
.
Compiling:
-en     .././simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     .././hal/hal.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/startup_sam4s.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/sam4s_hal.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/uart.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/pio.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/system_sam4s.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/sysclk.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/pmc.c ...
-e Done!
.
LINKING:
-en     sim

You should see a big list of `PLATFORM`s to build for. We left the `PLATFORM` blank in the command above, so the build system instead printed a list of supported platforms. Fill in your platform, rerun the build command, and your firmware should be successfully built.

Continuing on, there's two possible ways to upload firmware to your target:

1. ChipWhisperer has built in support for XMEGA, STM32F*, AVR, and SAM4S bootloaders. These can be used as follows

In [12]:
#cw.program_target(scope, cw.programmers.XMEGAProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.STM32FProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.AVRProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.SAM4SProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.STM32FProgrammer, "../firmware/mcu/simpleserial-base/simpleserial-base-CW308_STM32F3.hex")
#cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../firmware/mcu/simpleserial-base/simpleserial-base-CWHUSKY.hex")

cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../firmware/mcu/simpleserial-ml-kem-512/simpleserial-ml-kem-512-CWHUSKY.hex")

2. For other targets, you'll need to use an external programmer or a debugger to flash the firmware onto the target. 

Whatever your case, upload the firmware we built earlier to the target device. Next we'll be learning how to capture power traces and communicate with the target.

## Communication with the Target

Communication with targets, which is done through the `SimpleSerial target` object we got earlier, is grouped into two categories:

1. Raw serial via `target.read()`, `target.write()`, `target.flush()`, etc. 

1. SimpleSerial commands via `target.simpleserial_read()`, `target.simpleserial_write()`, `target.simpleserial_wait_ack()`, etc.

The firmware we uploaded uses the simpleserial protocol (https://wiki.newae.com/SimpleSerial), so we'll start off with simpleserial. Later, we'll use the raw serial commands to send the same messages.

If you check the simpleserial-base firmware (`simpleserial-base.c`) you'll find that for the simpleserial `'p'` command, the target will echo back the command. Let's try that out now:

In [ ]:
sk = bytearray([ 0x72, 0xC6, 0xF5, 0x69, 0x45, 0x31, 0xDF, 0x93, 0xBE, 0x7F, 0xFB, 0x37, 0xEF, 0x5D, 0xC9, 0x40, 0xD9, 0xC0, 0xB7, 0xD1, 0x21, 0x45, 0xBB, 0xFE, 0x17, 0x6E, 0x22, 0xBD, 0x82, 0xA4, 0xDC, 
0xEE, 0x79, 0x65, 0x43, 0xDD, 0x85, 0x1B, 0x1D, 0x43, 0x9C, 0xE8, 0x8, 0xAA, 0x53, 0x3D, 0x1F, 0x69, 0x24, 0x3A, 0xF0, 0x41, 0x2B, 0xFB, 0xC8, 0x8A, 0xF7, 0x33, 0x19, 0x88, 0x5, 0xCC, 0xFE, 
0x67, 0x9D, 0x47, 0xC2, 0x46, 0x80, 0xDC, 0xF6, 0xA0, 0xAF, 0x63, 0x18, 0xC5, 0xE3, 0x23, 0x1A, 0xE8, 0x6C, 0x57, 0xFE, 0xC6, 0xAB, 0xE1, 0x1F, 0xA7, 0x7B, 0xB6, 0xB4, 0xFE, 0xD, 0x83, 0xDA,
0xE1, 0xC, 0x7, 0x66, 0x3F, 0x69, 0x65, 0x5E, 0x6C, 0x41, 0xE3, 0xA4, 0x3E, 0xF8, 0xF6, 0xBF, 0x9, 0xC1, 0xEA, 0x84, 0x6D, 0xA1, 0x2D, 0x92, 0x6, 0x64, 0xFC, 0xD5, 0x24, 0x44, 0xD3, 0x83, 
0xCC, 0x71, 0x97, 0x78, 0xE0, 0x28, 0x9F, 0xDA, 0xC1, 0xE, 0xED, 0x91, 0xE2, 0xC1, 0xA2, 0xEF, 0x8B, 0x64, 0x90, 0x80, 0x8D, 0x92, 0x25, 0x9B, 0xEF, 0x5A, 0x4D, 0x67, 0x35, 0x4A, 0xA2, 0x71,
0xAA, 0xE, 0x99, 0x8, 0x37, 0x7C, 0xCF, 0xE0, 0xB5, 0xB4, 0x96, 0x8, 0xE9, 0x7E, 0x4, 0xB3, 0x4E, 0xB, 0x3A, 0x9D, 0xE8, 0xD5, 0x66, 0x6A, 0x1A, 0xF8, 0x9B, 0xDA, 0x2A, 0xD8, 0x5D, 0x2D, 
0x24, 0x79, 0x22, 0x96, 0x6, 0xDC, 0x9F, 0x9F, 0x65, 0x24, 0x7, 0x9D, 0x10, 0xE6, 0xE5, 0x5D, 0xD, 0x3D, 0x2D, 0xC1, 0xC2, 0x5B, 0x5C, 0x7E, 0x14, 0x83, 0x8F, 0xFF, 0x5, 0x77, 0x63, 0x3, 
0xB5, 0x50, 0x31, 0x29, 0x3E, 0xA9, 0x9C, 0x3C, 0xFF, 0x35, 0x14, 0xE1, 0x9A, 0x4A, 0x88, 0x71, 0xD2, 0xAC, 0x45, 0x7D, 0xC7, 0x9C, 0x58, 0xC9, 0xB5, 0x56, 0x8A, 0xC3, 0xFC, 0x9C, 0xA3, 
0x85, 0xFC, 0x1, 0x6A, 0xCE, 0x55, 0xA9, 0x8, 0xE0, 0xFA, 0xF2, 0x55, 0xE2, 0xE6, 0xFD, 0x8A, 0x39, 0x20, 0xFF, 0x52, 0xE3, 0x89, 0x77, 0x1A, 0x46, 0xE2, 0x8, 0x91, 0xBB, 0x8D, 0x84, 0xB5, 
0x55, 0x69, 0x3D, 0xAB, 0xDB, 0xF2, 0x1E, 0x5F, 0xA0, 0xAC, 0xF0, 0x3C, 0xF6, 0x90, 0x8A, 0x1F, 0x91, 0x5A, 0x2F, 0x91, 0x3A, 0x17, 0x3F, 0xD, 0xAD, 0xCE, 0xBE, 0x8E, 0x82, 0xEF, 0xB, 0xC9, 
0xDA, 0xC0, 0xDE, 0xD6, 0xF4, 0x1A, 0xEE, 0x78, 0x6F, 0xA1, 0x3C, 0x3C, 0xCD, 0x26, 0x8A, 0x80, 0x4D, 0x2F, 0x2, 0x43, 0x1, 0x58, 0xFC, 0x5C, 0x4C, 0xCE, 0x81, 0x9F, 0xD5, 0x16, 0xFA, 0x64, 
0x75, 0xE, 0x2, 0xA3, 0xA3, 0x48, 0x28, 0xD5, 0x54, 0x3B, 0x17, 0x7F, 0xBF, 0x38, 0x30, 0x8C, 0x46, 0x7E, 0xFD, 0x9E, 0x6C, 0x57, 0x6C, 0xD3, 0x3A, 0x3F, 0x80, 0x2D, 0x3D, 0x5D, 0x4D, 0xFE, 
0x8C, 0x97, 0x64, 0xCA, 0x69, 0x66, 0x1F, 0xDC, 0x74, 0x89, 0xEE, 0x67, 0x2D, 0xD6, 0x65, 0xCE, 0x98, 0xAE, 0x3E, 0xD, 0xB2, 0x58, 0xC0, 0xEF, 0xCC, 0x40, 0xB7, 0x5A, 0xA5, 0x46, 0x91, 0x78,
0xAF, 0x12, 0x6, 0xDB, 0x3F, 0x71, 0x37, 0xE3, 0xB9, 0xAA, 0x7A, 0x8A, 0x1F, 0x63, 0x93, 0x5F, 0xEB, 0xC3, 0x52, 0x58, 0x5D, 0x6E, 0x2D, 0x40, 0x81, 0x82, 0xA1, 0x97, 0x32, 0xCF, 0x1F, 0x2E,
0xEB, 0xC, 0x6A, 0x96, 0x4B, 0xB4, 0xB4, 0xDB, 0xC2, 0x19, 0x73, 0xB1, 0x34, 0x4B, 0x16, 0xF3, 0x52, 0xD8, 0x67, 0xCA, 0x40, 0xEC, 0x39, 0x65, 0x1E, 0xFC, 0x89, 0x23, 0xD8, 0x6B, 0xD5, 0x9B,
0x7D, 0xB2, 0xA, 0x71, 0xB0, 0xF2, 0x40, 0xE6, 0xCF, 0xA, 0x60, 0x66, 0xBC, 0x0, 0xD1, 0xEB, 0xAA, 0xD0, 0xF7, 0xDC, 0x2F, 0x20, 0xB6, 0x42, 0x3C, 0x46, 0xCD, 0xE5, 0x21, 0x83, 0x6A, 0x71, 
0x26, 0x25, 0x3E, 0x7E, 0x38, 0xCE, 0xCD, 0xD, 0xD7, 0xAD, 0xC9, 0x69, 0x9B, 0xF6, 0x8B, 0x37, 0xC1, 0xB1, 0xF1, 0x29, 0x9C, 0x69, 0x89, 0xCD, 0x68, 0x6E, 0xCA, 0x2D, 0x67, 0x4, 0x93, 0x3, 
0xDF, 0xC9, 0xF, 0x3B, 0xCF, 0xB8, 0x72, 0x0, 0x33, 0xE, 0xC4, 0x14, 0x8C, 0xA3, 0x2F, 0x7F, 0x97, 0x68, 0xB, 0x7B, 0x44, 0xF6, 0x71, 0xAF, 0x4F, 0xB6, 0x69, 0x22, 0x83, 0x4A, 0x23, 0x98, 
0x47, 0xA6, 0x14, 0x3F, 0xBD, 0x54, 0x53, 0x72, 0x67, 0x33, 0x5B, 0x97, 0x3C, 0x12, 0x77, 0x65, 0xE9, 0x4, 0x4D, 0xD8, 0xB7, 0x58, 0x47, 0x5, 0xFD, 0xEB, 0xA1, 0x3B, 0x38, 0xE8, 0xD9, 0x70, 
0x83, 0x4F, 0x5E, 0x73, 0x75, 0xBB, 0xC2, 0xFA, 0x78, 0x57, 0xFE, 0x37, 0xDB, 0x5B, 0x97, 0xBD, 0x74, 0x2C, 0xBE, 0xD8, 0xAF, 0x6D, 0x96, 0x96, 0x15, 0x70, 0x1A, 0xD8, 0x3A, 0x23, 0x5F, 
0xB8, 0x67, 0xCB, 0x77, 0xC, 0x72, 0x30, 0x64, 0x5E, 0x6A, 0x47, 0x56, 0x83, 0x51, 0xDA, 0x87, 0x82, 0x52, 0x1D, 0xD4, 0x7E, 0x2E, 0xF0, 0x85, 0x3C, 0xC0, 0x94, 0xBB, 0x30, 0x7B, 0xF2, 0xD9,
0xAD, 0x9A, 0x43, 0x6A, 0xC6, 0xA6, 0x22, 0xB6, 0xE2, 0x2A, 0x6, 0xDC, 0xD4, 0x1C, 0xA1, 0x1E, 0x69, 0x13, 0xF8, 0x39, 0xA, 0xB0, 0xEA, 0x6D, 0x9F, 0xE4, 0x1B, 0x91, 0xE9, 0x1D, 0xEE, 0xE7, 
0xB, 0xA, 0x77, 0xD1, 0x95, 0x1, 0xC1, 0xCF, 0x64, 0x76, 0x92, 0x2F, 0x55, 0x6C, 0x47, 0xEA, 0xCA, 0x6D, 0xB4, 0x63, 0xED, 0xF, 0x95, 0x86, 0xC1, 0x6, 0xF8, 0x41, 0x7A, 0x59, 0x63, 0xB5, 
0x94, 0x36, 0x1F, 0x7C, 0xA3, 0x98, 0xA7, 0x76, 0x8, 0xA7, 0x34, 0xD0, 0x48, 0x11, 0x3C, 0xC5, 0xF4, 0x2D, 0x48, 0xD9, 0x94, 0xF9, 0xF6, 0xEB, 0x80, 0x87, 0xFC, 0xDE, 0x3, 0x44, 0xE6, 0x23, 
0xF1, 0x82, 0xC8, 0x6B, 0x3D, 0x20, 0xC5, 0xAA, 0x13, 0xB2, 0x32, 0xDB, 0x8C, 0xAE, 0x85, 0x88, 0xF2, 0xB2, 0x6B, 0x78, 0xE8, 0xEA, 0xD9, 0x11, 0x2F, 0xB5, 0x80, 0x16, 0x15, 0x6E, 0x15, 
0x7D, 0xF6, 0x44, 0xE, 0xC7, 0xC3, 0x4E, 0x9A, 0x3A, 0x58, 0x28, 0x8F, 0x6D, 0xC9, 0xC1, 0xB2, 0x49, 0x7C, 0xA2, 0x65, 0x78, 0xAD, 0x72, 0x5D, 0xB6, 0x77, 0xC7, 0x38, 0x7, 0x70, 0xAF, 0xF5, 
0xE4, 0xF4, 0x1A, 0x8C, 0x14, 0x5B, 0x3E, 0xEA, 0xDC, 0xEF, 0xB5, 0xF8, 0x30, 0x62, 0x87, 0xDB, 0xC4, 0xE7, 0xF5, 0x8C, 0x4A, 0x12, 0xD, 0x66, 0xF5, 0x71, 0x6A, 0x87, 0x92, 0x28, 0xC, 0x5A, 
0x5A, 0xE6, 0xD, 0x87, 0xEB, 0xF, 0x3, 0x7F, 0x5D, 0xA5, 0xF2, 0xF4, 0xE9, 0xF4, 0xFC, 0x58, 0x81, 0xF8, 0x78, 0x48, 0x7D, 0x1D, 0x41, 0x66, 0x89, 0xD5, 0xBA, 0x8B, 0xC0, 0x64, 0xB1, 0x56, 
0x6D, 0x27, 0xEE, 0x40, 0x97, 0x2E, 0x97, 0xB2, 0x55, 0xA, 0x45, 0x60, 0x19, 0xC, 0x28, 0x50, 0xD, 0x5, 0xEC, 0x85, 0xB7, 0x56, 0x8D, 0xE2, 0x6B, 0xC, 0x1B, 0xCC, 0xE7, 0xAE, 0xC6, 0xAA, 
0x17, 0xD0, 0x69, 0x6D, 0xF4, 0x79, 0x44, 0xCE, 0xBE, 0xE, 0x8D, 0x5C, 0xCE, 0x67, 0xA5, 0xCC, 0x7, 0xDF, 0xF0, 0xA2, 0xDF, 0xDA, 0x9, 0x7D, 0xF5, 0x7A, 0xE2, 0x4B, 0x15, 0x7E, 0xC4, 0xC9, 
0xD9, 0x43, 0x28, 0xFF, 0xCA, 0xAB, 0x3D, 0x98, 0x2C, 0xF9, 0xFD, 0x67, 0xAE, 0x5B, 0x2E, 0xC4, 0x85, 0x17, 0x34, 0xCB, 0x72, 0x6E, 0x7C, 0x46, 0x6E, 0xDD, 0x99, 0x81, 0xE6, 0xF4, 0xCD, 
0x70, 0x3D, 0xC0, 0x7D, 0x83, 0xC6, 0x95, 0x44, 0x45, 0xB3, 0x26, 0xB3, 0xED, 0x6E, 0x55, 0xFC, 0xF7, 0xBB, 0x15, 0x4C, 0xB7, 0x38, 0x7C, 0x51, 0xC8, 0xE9, 0xF6, 0xEE, 0x32, 0xB6, 0x49, 
0xCC, 0x11, 0x20, 0xD9, 0x40, 0x39, 0x46, 0xBA, 0xDE, 0x4C, 0x6B, 0xD1, 0x95, 0x6A, 0xB8, 0x23, 0x20, 0xC5, 0xFA, 0x2E, 0xA6, 0x8B, 0xD, 0x61, 0x25, 0x4F, 0xE0, 0xE0, 0x26, 0x59, 0x21, 0xD7,
0x98, 0xFC, 0x56, 0x14, 0x32, 0xA1, 0x4, 0x55, 0x9E, 0x43, 0xB5, 0xFB, 0x2, 0x2A, 0xA5, 0x78, 0x3C, 0x23, 0xD6, 0xA6, 0x44, 0xCD, 0x6E, 0xD1, 0xB8, 0x85, 0x37, 0xB3, 0x42, 0x6, 0xEF, 0x86, 
0x6F, 0xB2, 0xC6, 0xF4, 0x7D, 0xF1, 0xFE, 0x30, 0xA8, 0xA5, 0x3A, 0x19, 0x4A, 0x11, 0x44, 0x90, 0xEA, 0x82, 0xCD, 0x41, 0xDF, 0xEA, 0x1E, 0x49, 0x27, 0x7F, 0x9F, 0x4C, 0xED, 0x16, 0x41, 
0xF2, 0x99, 0x78, 0xF1, 0x1, 0x2C, 0x67, 0x67, 0x9A, 0xD3, 0xE8, 0xC2, 0x7C, 0xA9, 0xFC, 0x77, 0xDE, 0xDC, 0x34, 0x4E, 0x25, 0xE6, 0xD5, 0x85, 0x2E, 0xB0, 0x6A, 0x1D, 0xB7, 0x70, 0x99, 0xB6,
0xF2, 0xCE, 0x9F, 0x90, 0xEB, 0x18, 0xB7, 0xC8, 0x8A, 0x19, 0xE9, 0xE1, 0xF7, 0xB, 0x12, 0xA5, 0x57, 0x1D, 0x5A, 0x77, 0xAF, 0x6D, 0x9C, 0x2F, 0x74, 0x8A, 0xE6, 0x2D, 0x81, 0x6, 0x29, 0xA4, 
0x47, 0x49, 0x52, 0xD3, 0x3A, 0x19, 0x48, 0xD5, 0xE8, 0x85, 0x66, 0xE4, 0x69, 0xF4, 0xB1, 0x1E, 0x1B, 0x26, 0xCE, 0x83, 0x42, 0x7A, 0x3E, 0x1, 0xB8, 0x55, 0x4, 0xAF, 0x51, 0x38, 0xEB, 0x40, 
0x9C, 0xE1, 0x31, 0x5E, 0xF7, 0xE6, 0xC5, 0x38, 0xF6, 0x57, 0x5C, 0x49, 0x5A, 0x1D, 0x61, 0xB1, 0x99, 0xB6, 0x4E, 0xF5, 0x6C, 0x7F, 0x6A, 0x81, 0x69, 0x34, 0x42, 0xEF, 0xA5, 0x3B, 0xF3, 
0x5F, 0x7C, 0xB3, 0xB4, 0x29, 0xBF, 0x84, 0xB3, 0x41, 0xE8, 0xE4, 0x83, 0x89, 0x5B, 0x64, 0x37, 0xBB, 0xBE, 0xCD, 0xD5, 0x70, 0x7A, 0x44, 0x6F, 0x42, 0x78, 0x91, 0x59, 0x8A, 0xD2, 0x4D, 
0xA0, 0x6D, 0x44, 0x24, 0x6E, 0xAB, 0x24, 0x17, 0xB0, 0x2, 0x1D, 0x3B, 0xB6, 0xC5, 0x14, 0x2B, 0xB0, 0x4F, 0x82, 0x1B, 0x5C, 0x3F, 0x89, 0xD8, 0x7B, 0xC9, 0xC4, 0x92, 0xA0, 0x97, 0xA2, 0x5D,
0x6F, 0x12, 0xBD, 0xE0, 0xEE, 0x9A, 0xE5, 0x51, 0x36, 0x14, 0x78, 0x6D, 0x56, 0x5F, 0x10, 0xCA, 0x49, 0x6, 0xB1, 0x34, 0x95, 0x44, 0xF8, 0xD3, 0x7B, 0x30, 0xD1, 0xE5, 0xC5, 0x1C, 0xC1, 0xCA,
0x4C, 0x47, 0x7C, 0x85, 0xBB, 0xAB, 0x19, 0x16, 0xD6, 0x17, 0xC9, 0x8D, 0x35, 0x84, 0x11, 0x74, 0xBC, 0xFA, 0xFB, 0xFE, 0x4A, 0x25, 0x8C, 0x57, 0x88, 0xF9, 0x8, 0x8B, 0xAF, 0xE0, 0x4A, 0xD1,
0x4E, 0x2C, 0xB8, 0xEE, 0xB3, 0xFA, 0x4C, 0xC0, 0xBA, 0x83, 0x9, 0xE7, 0x43, 0xE6, 0xEE, 0xA2, 0x26, 0x1C, 0xFF, 0x78, 0xEE, 0x74, 0x20, 0x89, 0xF6, 0x49, 0xD0, 0x4D, 0x31, 0xF3, 0x3D, 0xA6,
0x77, 0x77, 0x8C, 0xF8, 0x48, 0x9A, 0x54, 0x79, 0xF8, 0x6F, 0x37, 0x78, 0x5C, 0xA9, 0x36, 0x9C, 0x5F, 0x71, 0x44, 0x40, 0x7C, 0x69, 0x67, 0x80, 0xDD, 0xE, 0x5A, 0xAC, 0xFD, 0x97, 0xC9, 0x7B,
0x65, 0xFC, 0x57, 0xA6, 0xE6, 0xAD, 0x18, 0xCF, 0xFA, 0x52, 0x69, 0xC8, 0x15, 0x66, 0x18, 0x70, 0x5D, 0x98, 0x1C, 0xE6, 0xB9, 0xC5, 0xE6, 0xC2, 0xD2, 0xEF, 0xC4, 0x9D, 0xA1, 0x1F, 0x7, 0x80,
0xE5, 0xD4, 0x31, 0x4E, 0x44, 0x3E, 0xFE, 0xB, 0x3C, 0xF0, 0xF6, 0xC3, 0x14, 0x30, 0xBB, 0x8D, 0x2E, 0x6D, 0x1F, 0x42, 0xB6, 0x9E, 0x59, 0xF9, 0x9A, 0x7, 0x2F, 0xEB, 0xB7, 0xD9, 0x88, 0xCC, 
0x57, 0xD9, 0xC6, 0x8, 0xD, 0x4A, 0x38, 0x7D, 0x9A, 0x66, 0x7A, 0xBD, 0x5A, 0xD6, 0x71, 0xC8, 0x4F, 0x39, 0xCA, 0x67, 0x6, 0xEF, 0x10, 0xE3, 0x52, 0x75, 0x60, 0x45, 0x65, 0x2D, 0xA, 0xFB, 
0x38, 0x52, 0xD4, 0xB, 0x24, 0x5B, 0xB7, 0xFD, 0x1, 0x5B, 0x96, 0x8B, 0x7E, 0x15, 0x48, 0x1, 0xF6, 0xCD, 0xC8, 0x8A, 0xBB, 0x8F, 0xFF, 0xB9, 0xCA, 0x96, 0xF1, 0xE9, 0xF, 0x43, 0xBB, 0xB1, 
0x37, 0xCF, 0x10, 0xE, 0x3A, 0x6C, 0x17, 0x8B, 0xE8, 0x39, 0xFC, 0xDE, 0x23, 0xBD, 0xA1, 0x30, 0x9A, 0xB9, 0x72, 0xE5, 0xA7, 0x8F, 0x8F, 0x8, 0xB5, 0xA, 0xBC, 0x75, 0xBC, 0xA8, 0xE2, 0x62, 
0x2F, 0x6C, 0x35, 0xC7, 0xFB, 0xB1, 0xF, 0xA5, 0x2F, 0x9C, 0x30, 0xDB, 0xB1, 0x65, 0xC0, 0x65, 0x25, 0x21, 0x6D, 0x68, 0x89, 0x8B, 0xB3, 0xEA, 0x21, 0x35, 0xDA, 0x7D, 0xE9, 0x73, 0xF1, 0xDE,
0xB, 0x48, 0xD3, 0x3A, 0xD1, 0x1, 0x31, 0x55, 0x98 ])

pk = bytearray([ 0x5, 0xB6, 0xA6, 0xA6, 0x69, 0x2D, 0xED, 0x56, 0x2E, 0x67, 0xC3, 0x9C, 0xF1, 0xD5, 0xA6, 0x17, 0xC7, 0x57, 0x5, 0xFD, 0x43, 0x4B, 0xB8, 0xD6, 0xC5, 0x79, 0x5E, 0xD4, 0xA4, 0xCB, 0xA8, 
0x7E, 0xDB, 0xF8, 0x4D, 0x6, 0x91, 0x78, 0xAE, 0x52, 0x9, 0x4D, 0xE4, 0x42, 0xEA, 0xDE, 0x2C, 0x97, 0xD1, 0xC3, 0xDE, 0xAF, 0x1E, 0xBA, 0x64, 0x61, 0xC6, 0x8D, 0x35, 0x1D, 0x35, 0x74, 0x4A, 
0xB0, 0x7, 0x60, 0x14, 0x5, 0x23, 0xCB, 0xEA, 0x67, 0xC7, 0x6B, 0xC, 0xB2, 0x30, 0xD9, 0xEE, 0x34, 0xF7, 0x8C, 0x70, 0x37, 0xBD, 0x92, 0x40, 0x2A, 0xAC, 0xC2, 0x68, 0x41, 0xB7, 0x3C, 0xB, 
0x74, 0x1A, 0xB5, 0x74, 0x5C, 0x49, 0x98, 0xC5, 0x47, 0xCC, 0x3B, 0x5E, 0x3F, 0x3A, 0x3F, 0x4C, 0xDC, 0xF1, 0x89, 0x6B, 0x2B, 0x65, 0x4D, 0xE0, 0xDF, 0xAF, 0x2C, 0xD6, 0x9E, 0x9, 0xCF, 0x37,
0x58, 0x47, 0x8A, 0x8B, 0x3B, 0x24, 0x9, 0x5C, 0x7, 0xDD, 0x34, 0x9A, 0x5E, 0xCB, 0x4F, 0x2D, 0x2B, 0xCA, 0x91, 0xED, 0xD4, 0xE, 0xFA, 0x3F, 0xF5, 0x67, 0xA4, 0x30, 0xBE, 0x89, 0x1A, 0x8F, 
0x16, 0x87, 0x3D, 0x85, 0x9, 0xA9, 0xAC, 0xFB, 0xE8, 0xB3, 0x7D, 0xF, 0x26, 0x98, 0x6A, 0xD9, 0x37, 0xDC, 0x7, 0xFC, 0xA3, 0xE, 0x5, 0x96, 0xC5, 0xA6, 0x7D, 0x52, 0xE0, 0xCD, 0x7E, 0x2F, 
0x85, 0xD4, 0xC2, 0xC7, 0xB8, 0xFF, 0x2A, 0xF, 0xAA, 0xF6, 0x98, 0x78, 0x2B, 0x44, 0xCE, 0x7D, 0xD8, 0x66, 0x63, 0x3B, 0x63, 0xAA, 0xEC, 0x80, 0x81, 0x8E, 0x56, 0x7, 0x60, 0xD9, 0x14, 0x23, 
0xE4, 0x11, 0x3B, 0xBC, 0x5C, 0x8A, 0x82, 0x7A, 0x2F, 0x59, 0x9, 0x41, 0x62, 0x91, 0x6A, 0x62, 0x68, 0x33, 0x37, 0x17, 0x39, 0x30, 0x63, 0x56, 0x95, 0x5E, 0x0, 0x51, 0x34, 0xBE, 0x5C, 0x9D, 
0x1D, 0x92, 0x2A, 0xCC, 0x6, 0x65, 0xC1, 0x2D, 0x3C, 0xDC, 0xD2, 0x94, 0x2F, 0x66, 0x7A, 0xB9, 0x2, 0x92, 0xDC, 0xC, 0xE2, 0x0, 0xAE, 0x1A, 0xBC, 0x16, 0x58, 0x23, 0x22, 0xCA, 0x8B, 0xEE, 
0xBC, 0x4E, 0x9C, 0x65, 0xEF, 0x29, 0x56, 0x34, 0x39, 0xFA, 0x3D, 0xBC, 0x62, 0xB2, 0x5B, 0xF4, 0x5D, 0x5C, 0x43, 0xE3, 0xA, 0x42, 0xAD, 0xE1, 0x31, 0x89, 0x29, 0xA6, 0x8A, 0x32, 0xDD, 0xED,
0xD6, 0x8F, 0x8B, 0x54, 0x90, 0x64, 0xD3, 0xEF, 0x15, 0x68, 0x9B, 0xC7, 0xE9, 0xCD, 0xA3, 0x73, 0x57, 0x1A, 0xF7, 0xBA, 0x39, 0xD3, 0x22, 0xC, 0xEA, 0xEB, 0x1, 0x1A, 0xFE, 0xE, 0xDA, 0x5B, 
0x1A, 0x4F, 0xC0, 0x71, 0xBE, 0xC5, 0x81, 0x21, 0x4F, 0x60, 0x27, 0x11, 0xD0, 0xD1, 0xB2, 0xF1, 0x85, 0xFD, 0xDE, 0x6C, 0x5A, 0x38, 0x64, 0xEB, 0x7, 0xAD, 0x8, 0xB0, 0x8C, 0x6B, 0xBC, 0xCF, 
0x65, 0xC7, 0x2D, 0xE0, 0x48, 0x2F, 0x9D, 0xD3, 0x7F, 0x5F, 0x21, 0xC, 0x7C, 0xCE, 0x7A, 0x1, 0x90, 0xD1, 0x9C, 0x8E, 0x43, 0x75, 0x1E, 0xF5, 0xED, 0x24, 0x63, 0x20, 0xD, 0x68, 0x87, 0xFB, 
0xAD, 0x1C, 0x6A, 0x1D, 0xDC, 0xDB, 0x91, 0xF1, 0x7D, 0xB9, 0x44, 0xE6, 0x91, 0xF, 0x88, 0xE0, 0x29, 0x5E, 0xB8, 0xFF, 0xE3, 0xF3, 0xE8, 0x1, 0x15, 0x64, 0xD1, 0xBD, 0xC4, 0xFF, 0x5D, 0x5C, 
0x2A, 0xED, 0xAC, 0xD8, 0xF3, 0xB4, 0xDA, 0x5E, 0x4, 0xCE, 0xD6, 0xAC, 0x55, 0xE2, 0x7, 0x8C, 0x5, 0x50, 0x3F, 0xF2, 0x5, 0xE7, 0xEF, 0x57, 0xA1, 0xE4, 0xB5, 0x24, 0xE8, 0x2C, 0x39, 0x57, 
0xFA, 0x5C, 0xF2, 0xE0, 0xF7, 0x72, 0xC4, 0x7B, 0xA0, 0x3A, 0xE, 0xB1, 0x28, 0x70, 0x5F, 0x39, 0xB2, 0x92, 0x42, 0x45, 0x78, 0x11, 0x79, 0x31, 0x95, 0xEE, 0xA4, 0xC6, 0x5D, 0x4D, 0x67, 0xD2,
0x16, 0xF2, 0x25, 0x5B, 0x50, 0x1E, 0x98, 0xFA, 0xE3, 0xFF, 0xE1, 0xC2, 0x4A, 0xC0, 0x64, 0xB5, 0xC5, 0x68, 0xC8, 0xA9, 0x21, 0xFB, 0x26, 0x86, 0xD, 0x1D, 0x4F, 0x9E, 0x56, 0x19, 0xC2, 0x29,
0x74, 0xED, 0x49, 0xFB, 0xB1, 0x6C, 0x4, 0xE7, 0x0, 0x17, 0xF0, 0x73, 0xA8, 0x49, 0xE5, 0x5E, 0xB8, 0x70, 0x9E, 0xE7, 0x2B, 0xB, 0x9F, 0xC4, 0xE5, 0xDD, 0xFC, 0x92, 0x4A, 0xA2, 0xA6, 0x87, 
0xAC, 0x25, 0x52, 0x70, 0x45, 0xDA, 0xF2, 0x5B, 0xD8, 0xA5, 0xB1, 0xD3, 0xF6, 0x5B, 0xFC, 0xAC, 0x2A, 0xAB, 0x12, 0xC0, 0x28, 0x92, 0xB4, 0xAE, 0x47, 0x43, 0x7F, 0x59, 0x86, 0xEE, 0xD3, 
0xC2, 0x90, 0x12, 0xCC, 0x25, 0x7F, 0x1E, 0xD7, 0xD5, 0x56, 0x16, 0x1E, 0x4, 0x49, 0xEF, 0x8C, 0x32, 0xFC, 0xD5, 0xBD, 0xBD, 0x79, 0xE6, 0x4D, 0xBE, 0x78, 0xFE, 0xAA, 0x6A, 0x52, 0xBA, 0x43,
0x94, 0x62, 0xC4, 0x97, 0x11, 0xBD, 0x19, 0x4D, 0x25, 0x6C, 0x2B, 0xDA, 0x7B, 0xA5, 0x2B, 0x2B, 0xF7, 0x8A, 0x96, 0x76, 0x22, 0xEB, 0xF4, 0x95, 0xEF, 0x2B, 0x64, 0x84, 0x29, 0xE1, 0x82, 
0xC0, 0x5C, 0xB9, 0xE3, 0x9F, 0xDD, 0x71, 0x96, 0xA7, 0x2B, 0x7A, 0x3B, 0x9C, 0xFE, 0x5E, 0x6, 0x4C, 0x67, 0x50, 0xBF, 0xD9, 0x44, 0x43, 0xEE, 0xC3, 0x11, 0xA1, 0x97, 0x43, 0x6B, 0xD3, 0xA3,
0x3, 0xF2, 0x3B, 0x65, 0x14, 0xD4, 0x2C, 0x2D, 0xF7, 0x14, 0xFF, 0xB9, 0x92, 0x2B, 0x6D, 0x83, 0x45, 0x55, 0xCF, 0x56, 0x47, 0x24, 0x94, 0x6E, 0xC6, 0xCB, 0x4D, 0xB, 0x4C, 0xF, 0xC8, 0x3E, 
0x9F, 0xB3, 0x14, 0x21, 0xF0, 0x32, 0x47, 0x9E, 0xC2, 0xFB, 0x4D, 0x97, 0x7, 0xDF, 0x27, 0x1B, 0xD5, 0x82, 0xB4, 0xAB, 0x2A, 0x21, 0xB8, 0xE8, 0xF6, 0x40, 0xE, 0x8, 0xFC, 0x8C, 0x2C, 0xD3, 
0x4, 0xCB, 0x76, 0x3C, 0x61, 0x44, 0xBB, 0x7F, 0x6B, 0x90, 0xA8, 0xC7, 0xA7, 0x13, 0x15, 0x71, 0x98, 0x3, 0xF6, 0x60, 0xBA, 0xDD, 0x75, 0xE0, 0x23, 0x77, 0xEB, 0x70, 0x33, 0xDF, 0x68, 0xBD, 
0x49, 0xDB, 0xB4 ]);

ct = bytearray([0x8f, 0x77, 0xf3, 0x23, 0x98, 0xa0, 0x05, 0xa8, 0x83, 0x38, 0xb0, 0xb8, 0x0c, 0x6c, 0x01, 0x25, 
0xe8, 0x2d, 0x80, 0x83, 0xad, 0x00, 0x96, 0x18, 0x99, 0xf0, 0x3f, 0x8d, 0x91, 0x19, 0x80, 0x34, 
0x3e, 0x00, 0x6f, 0xc6, 0x70, 0x32, 0x7e, 0x59, 0x14, 0x41, 0x1d, 0x5e, 0xa2, 0xfd, 0x8a, 0x45, 
0xf7, 0xe5, 0xf7, 0xd0, 0x97, 0xaa, 0xb4, 0x8d, 0xb5, 0x67, 0x39, 0xbc, 0x2c, 0xd3, 0xf1, 0xc0, 
0x77, 0x68, 0x47, 0x31, 0x1f, 0x90, 0xe9, 0x70, 0x10, 0xfe, 0xe5, 0x06, 0x22, 0x40, 0x9c, 0x55, 
0x2d, 0xcc, 0x46, 0x00, 0x66, 0xa1, 0x18, 0x6c, 0x44, 0x7f, 0x10, 0x3e, 0x2c, 0x68, 0x92, 0xa4, 
0xc5, 0xe1, 0x46, 0x32, 0x0b, 0x46, 0x1b, 0x54, 0xb5, 0x88, 0xaa, 0x17, 0x24, 0xc7, 0xb7, 0xa7, 
0xd9, 0x79, 0xab, 0x72, 0x9b, 0x21, 0x6c, 0xfb, 0xb1, 0xe5, 0x52, 0x05, 0xfa, 0xc5, 0x52, 0x1e, 
0xf4, 0xf8, 0xfc, 0x46, 0x82, 0x9a, 0x83, 0x3a, 0x29, 0xa6, 0xcb, 0xd7, 0xc5, 0x7a, 0x63, 0x04, 
0x9f, 0x54, 0xeb, 0x11, 0x90, 0x16, 0x54, 0xe1, 0x96, 0x20, 0xc6, 0x79, 0x67, 0xce, 0x44, 0x98, 
0x2d, 0xc1, 0x6e, 0x78, 0xf0, 0x65, 0xe4, 0x18, 0xdf, 0x18, 0x41, 0xf9, 0x96, 0x55, 0xb8, 0x44, 
0x62, 0x21, 0x7b, 0xe0, 0x76, 0x80, 0xb3, 0x6d, 0x1a, 0x95, 0xb8, 0xa8, 0x3b, 0xdf, 0x39, 0x80, 
0xd5, 0x8f, 0x42, 0x32, 0x4c, 0xad, 0x5f, 0xd3, 0xd3, 0x31, 0xfb, 0x48, 0xa5, 0x97, 0x9e, 0x07, 
0xa2, 0x24, 0xf1, 0xc0, 0xc7, 0xb3, 0x41, 0x69, 0xb9, 0x8e, 0xea, 0xa4, 0x3d, 0x76, 0xf6, 0x23, 
0xd4, 0xeb, 0xd3, 0x64, 0xe5, 0xc8, 0x0d, 0x1f, 0x53, 0x4d, 0x26, 0xd7, 0xff, 0x1b, 0x10, 0xaa, 
0x90, 0x87, 0x11, 0xc9, 0xe5, 0xa6, 0x5e, 0x96, 0xd6, 0x80, 0xad, 0xf6, 0xc8, 0x33, 0x9c, 0x28, 
0xa5, 0xf2, 0x7a, 0x6c, 0x4a, 0xa6, 0x84, 0xe4, 0xba, 0x36, 0x9e, 0x14, 0x55, 0x9a, 0x9e, 0x14, 
0xb8, 0x6e, 0xb5, 0xb7, 0x54, 0x97, 0x15, 0x60, 0x65, 0x37, 0xec, 0xdc, 0xe3, 0xc2, 0xd0, 0x99, 
0xef, 0x64, 0xa5, 0x3c, 0x5d, 0x0b, 0x71, 0x47, 0x1e, 0x1a, 0x1a, 0xc7, 0x89, 0xf1, 0x17, 0xc9, 
0x7a, 0x4f, 0x83, 0x30, 0x72, 0x76, 0xb3, 0xdc, 0x6c, 0xca, 0x46, 0x40, 0x32, 0xc4, 0xa8, 0x39, 
0x69, 0xd2, 0xad, 0x96, 0xfd, 0x05, 0xf2, 0x9c, 0x47, 0x27, 0xe3, 0x85, 0xcd, 0x62, 0x1a, 0x97, 
0x7a, 0xf7, 0xf9, 0xac, 0x19, 0xe1, 0x3c, 0x44, 0x38, 0xec, 0x71, 0x6e, 0xa7, 0x51, 0x7d, 0x2d, 
0xe0, 0x9c, 0x04, 0xfd, 0xc5, 0x75, 0x9c, 0x14, 0xc8, 0x3c, 0x79, 0x93, 0xd7, 0xfd, 0xca, 0xe9, 
0x83, 0x3e, 0xe9, 0x71, 0xc9, 0x15, 0x3e, 0x80, 0x8c, 0xe1, 0x82, 0x8f, 0x85, 0x1b, 0xce, 0x6c, 
0x29, 0x74, 0x7f, 0xc1, 0x59, 0x63, 0x0e, 0xa5, 0xea, 0xfd, 0x2d, 0x60, 0x31, 0xb4, 0x4f, 0x93, 
0x66, 0x38, 0x4f, 0xe4, 0x7d, 0xc6, 0x9b, 0xf1, 0x2b, 0xf1, 0x9e, 0x8c, 0x5a, 0x90, 0x43, 0xb0, 
0x8d, 0x2a, 0x85, 0x29, 0x68, 0xc8, 0x07, 0xa7, 0xbc, 0xb9, 0x68, 0xb0, 0x34, 0xe4, 0xbf, 0x08, 
0x31, 0x8f, 0x41, 0x5a, 0x3e, 0xe4, 0x63, 0xed, 0xef, 0xca, 0xfa, 0x1f, 0xf9, 0xa0, 0x9b, 0xdd, 
0xac, 0x8f, 0x88, 0xc4, 0xe8, 0x17, 0xf5, 0x63, 0x4e, 0x65, 0x2a, 0xe3, 0x95, 0x7e, 0x8b, 0xcd, 
0x73, 0x2c, 0x2a, 0xa4, 0x6b, 0xc9, 0x6c, 0x78, 0x75, 0x4d, 0xd3, 0x30, 0xbd, 0x73, 0x49, 0x9b, 
0x90, 0x5f, 0xe8, 0x7d, 0xc4, 0x9f, 0xbe, 0x89, 0x53, 0x86, 0x5d, 0x04, 0x12, 0x6c, 0x39, 0xb8, 
0xd1, 0x02, 0x2e, 0x04, 0xdc, 0x02, 0xeb, 0x69, 0x4f, 0x7b, 0xc6, 0xd8, 0x88, 0x91, 0xb9, 0x53, 
0x1e, 0x95, 0x6b, 0xee, 0xab, 0xb1, 0xc8, 0xea, 0x83, 0x04, 0xa8, 0x04, 0x69, 0x73, 0x0d, 0xfa, 
0x52, 0x92, 0x6d, 0x31, 0x34, 0xf3, 0xd3, 0x9a, 0x4d, 0x55, 0x14, 0xe1, 0x88, 0x88, 0xe4, 0x61, 
0xfc, 0x7d, 0x41, 0xff, 0xfc, 0x9f, 0xdc, 0x93, 0xec, 0x17, 0x0a, 0xae, 0x4c, 0x4a, 0x60, 0x42, 
0xcf, 0x3e, 0x2d, 0xfc, 0x43, 0x7c, 0x2d, 0xfc, 0x83, 0x38, 0x2b, 0xfc, 0xbc, 0x14, 0xff, 0xfc, 
0xbb, 0xc1, 0xeb, 0x6c, 0xf4, 0x70, 0x4f, 0x9c, 0xd0, 0xf7, 0xff, 0xc8, 0x6f, 0xf4, 0x79, 0xd5, 
0x37, 0xa6, 0xe7, 0xf7, 0x26, 0x22, 0x14, 0x70, 0xe5, 0x92, 0x71, 0x60, 0x3c, 0xd1, 0x43, 0x74, 
0xae, 0xd8, 0x99, 0xb5, 0xee, 0xe3, 0xfa, 0x76, 0x60, 0x8e, 0xb0, 0xe7, 0xf3, 0x20, 0x7b, 0x6c, 
0xeb, 0xe9, 0xc1, 0x39, 0xac, 0xbc, 0x38, 0x02, 0x2c, 0x5a, 0x44, 0xf7, 0xf2, 0xd2, 0xe5, 0xad, 
0x48, 0x9c, 0x6d, 0x17, 0xe7, 0x45, 0x9b, 0x73, 0xde, 0xc3, 0xd6, 0xc5, 0xf1, 0xbd, 0x86, 0x35, 
0x6d, 0x52, 0x00, 0x0d, 0x44, 0xb2, 0x37, 0xd2, 0xa2, 0xbe, 0xf6, 0x85, 0x59, 0x65, 0x4b, 0x5d, 
0xdf, 0xa1, 0xb2, 0xd4, 0xdf, 0x50, 0xe0, 0x00, 0xe5, 0x91, 0x97, 0xa7, 0xe3, 0xe6, 0x15, 0x7a, 
0x63, 0x20, 0x9e, 0xf0, 0x7a, 0x8c, 0xfe, 0xf4, 0x00, 0xa0, 0xcd, 0xe2, 0xda, 0xcc, 0xb0, 0xce, 
0x18, 0x04, 0xdf, 0xc0, 0x29, 0xfe, 0x5e, 0x08, 0xa9, 0x1e, 0x8f, 0x3a, 0x9e, 0xa0, 0xb0, 0x11, 
0xcd, 0x1a, 0x82, 0x87, 0xdb, 0x27, 0xd7, 0x84, 0x3a, 0x2d, 0x0b, 0x71, 0x6a, 0x6e, 0x4f, 0x48, 
0x50, 0x11, 0xbb, 0x7f, 0x21, 0x04, 0xca, 0x09, 0x8a, 0x9d, 0xe7, 0xad, 0x50, 0x9f, 0xd2, 0x44, 
0xb1, 0x35, 0x8d, 0xb7, 0xd2, 0x80, 0xd6, 0xfa, 0xc7, 0xc0, 0xad, 0xdc, 0x58, 0xc8, 0x51, 0x76])

ss = bytearray([ 0xD9, 0xE6, 0x83, 0x63, 0x37, 0xF9, 0xD8, 0x9C, 0xBC, 0x8C, 0xBC, 0xC, 0x71, 0xEF, 0x2, 0xB5, 0x3E, 0x6D, 0x3C, 0x76, 0xF1, 0x3D, 0x2E, 0xD9, 0xD3, 0x20, 0x85, 0x5B, 0x5, 0x31, 0x80, 0x45 
])

In [ ]:
#Tenting secret key
SK_LEN = 1632
# The chunk size can be up to 100 bytes in SAM4S
# For STM32 seze upto 240 bytes,
# Also the size can be go from 1-240 bytes without any issues
# But in some cases it's just a size adjustment.
chunk_size = 102  # Keep <= 100 to reduce the risk of UART overrun on SAM4S
num_chunks = (SK_LEN + chunk_size - 1) // chunk_size

# Limpiar basura de ejecuciones anteriores para evitar desincronizacion del parser
# Clean the trash from previous executions and avoid parser desynchronization
target.reset_comms()
target.flush()

# Verificar comandos disponibles en el firmware cargado
# Verify vailable commands in the loaded firmware
cmds = target.get_simpleserial_commands()
print("Reported comands:", cmds)

# 1) Configurar chunk size en firmware y esperar ACK
# 1) Configure chunk size in the firmware and wait ACK
target.send_cmd(0x04, 0, [chunk_size & 0xFF, (chunk_size >> 8) & 0xFF])
ack = target.simpleserial_wait_ack()
print(f"ACK 0x04: {ack}")
if ack is None or len(ack) == 0 or ack[0] != 0x00:
    raise RuntimeError(f"Fallo en cmd 0x04, ack={ack}")
time.sleep(0.05)

In [ ]:
# 2) Enviar SK en chunks con ACK por cada envio
# 2) Send the secret key using chunks.
for i in range(num_chunks):
    start = i * chunk_size
    end = min(start + chunk_size, SK_LEN)
    tx_chunk = sk[start:end]
    target.send_cmd(0x02, i, tx_chunk)
    ack = target.simpleserial_wait_ack()
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ERROR sending chunk {i}: ack={ack}")
    print(f"Sent chunk {i}/{num_chunks-1}: {len(tx_chunk)} bytes")
    time.sleep(0.02)

In [ ]:
# 3) Leer cada chunk: primero paquete 'k', luego ACK del comando 0x03
# 3) Read each chunk: 1st from the paket 'k', then the ACK from the comand 0x03
rec_sk = bytearray()
target.con()
for i in range(num_chunks):
    start = i * chunk_size
    expected_len = min(chunk_size, SK_LEN - start)

    target.send_cmd(0x03, i, [])

    # Leer respuesta de datos sin consumir ACK automaticamente
    # Read response from data without automaic ACK consumption
    rx_chunk = target.simpleserial_read('k', expected_len, ack=False)
    if rx_chunk is None:
        raise RuntimeError(f"Timeout reading chunk {i}")

    ack = target.simpleserial_wait_ack()
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ERROR ACK reading chunk {i}: ack={ack}")

    rec_sk.extend(rx_chunk)
    print(f"Receiving chunk {i}/{num_chunks-1}: {len(rx_chunk)} bytes")
    time.sleep(0.02)


In [ ]:
print("\n--- Verify ---")
if len(rec_sk) != len(sk):
    print(f"ERROR: Length mismatch. Expected: {len(sk)}, Received: {len(rec_sk)}")
else:
    if rec_sk == sk:
        print("Success: Received key same as the Sended ")
    else:
        print("ERROR: Key misstach ")
        # Encontrar primera diferencia
        # Find the first difference
        for i in range(len(sk)):
            if sk[i] != rec_sk[i]:
                print(f"First difference in byte {i}: expected 0x{sk[i]:02x}, received 0x{rec_sk[i]:02x}")
                break

In [ ]:
#PK_LEN = 800 test
# The chunk size can be up to 100 bytes in SAM4S
# For STM32 seze upto 240 bytes,
# Also the size can be go from 1-240 bytes without any issues
# But in some cases it's just a size adjustment.
PK_LEN = 800
chunk_size = 100   #Keep <= 100 to reduce the risk of UART overrun on SAM4S
num_chunks = (PK_LEN + chunk_size - 1) // chunk_size

# Limpiar basura de ejecuciones anteriores para evitar desincronizacion del parser
#Clean the trash from previous executions and avoid parser desynchronization
target.reset_comms()
target.flush()

# Verificar comandos disponibles en el firmware cargado
# Verify vailable commands in the loaded firmware
cmds = target.get_simpleserial_commands()
print("Reported comands:", cmds)

# 1) Configurar chunk size en firmware y esperar ACK
# 1) Configure chunk size in the firmware and wait ACK
target.send_cmd(0x04, 0, [chunk_size & 0xFF, (chunk_size >> 8) & 0xFF])
ack = target.simpleserial_wait_ack()
print(f"ACK 0x04: {ack}")
if ack is None or len(ack) == 0 or ack[0] != 0x00:
    raise RuntimeError(f"Fallo en cmd 0x04, ack={ack}")
time.sleep(0.05)

In [ ]:
# 2) Enviar Pk en chunks con ACK por cada envio
# 2) Send the secret key using chunks.
for i in range(num_chunks):
    start = i * chunk_size
    end = min(start + chunk_size, PK_LEN)
    tx_chunk = pk[start:end]
    target.send_cmd(0x05, i, tx_chunk)
    ack = target.simpleserial_wait_ack()
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ERROR sending chunk {i}: ack={ack}")
    print(f"Sent chunk {i}/{num_chunks-1}: {len(tx_chunk)} bytes")
    time.sleep(0.02)

In [ ]:
# 3) Leer cada chunk: primero paquete 'p', luego ACK del comando 0x06
# 3) Read each chunk: 1st from the paket 'p', then the ACK from the comand 0x06
rec_pk = bytearray()
target.con()
for i in range(num_chunks):
    start = i * chunk_size
    expected_len = min(chunk_size, PK_LEN - start)

    target.send_cmd(0x06, i, [])

    # Leer respuesta de datos sin consumir ACK automaticamente
    # Read response from data without automaic ACK consumption
    rx_chunk = target.simpleserial_read('p', expected_len, ack=False)
    if rx_chunk is None:
        RuntimeError(f"Timeout reading chunk {i}")

    ack = target.simpleserial_wait_ack()
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ERROR ACK reading chunk {i}: ack={ack}")

    rec_pk.extend(rx_chunk)
    print(f"Receiving chunk {i}/{num_chunks-1}: {len(rx_chunk)} bytes")
    time.sleep(0.02)

In [ ]:
print("\n--- Verify ---")
if len(rec_pk) != len(pk):
    print(f"ERROR: Length mismatch. Expected: {len(pk)}, Received: {len(rec_pk)}")
else:
    if rec_pk == pk:
        print("Success: Received key same as the Sended ")
    else:
        print("ERROR: Key misstach ")
        # Encontrar primera diferencia
        # Find the first difference
        for i in range(len(pk)):
            if pk[i] != rec_pk[i]:
                print(f"First difference in byte {i}: expected 0x{pk[i]:02x}, received 0x{rec_pk[i]:02x}")
                break

In [ ]:
#CT_LEN = 768 test
CT_LEN = 768
# The chunk size can be up to 100 bytes in SAM4S
# For STM32 seze upto 240 bytes,
# Also the size can be go from 1-240 bytes without any issues
# But in some cases it's just a size adjustment.
chunk_size = 102  # Keep <= 100 to reduce the risk of UART overrun on SAM4S
num_chunks = (CT_LEN + chunk_size - 1) // chunk_size

# Limpiar basura de ejecuciones anteriores para evitar desincronizacion del parser
# Clean the trash from previous executions and avoid parser desynchronization
target.reset_comms()
target.flush()

# Verificar comandos disponibles en el firmware cargado
# Verify vailable commands in the loaded firmware
cmds = target.get_simpleserial_commands()
print("Reported comands:", cmds)

# 1) Configurar chunk size en firmware y esperar ACK
# 1) Configure chunk size in the firmware and wait ACK
target.send_cmd(0x04, 0, [chunk_size & 0xFF, (chunk_size >> 8) & 0xFF])
ack = target.simpleserial_wait_ack()
print(f"ACK 0x04: {ack}")
if ack is None or len(ack) == 0 or ack[0] != 0x00:
    raise RuntimeError(f"Fallo en cmd 0x04, ack={ack}")
time.sleep(0.05)

In [ ]:
# 2) Enviar SK en chunks con ACK por cada envio
# 2) Send the secret key using chunks.
for i in range(num_chunks):
    start = i * chunk_size
    end = min(start + chunk_size, CT_LEN)
    tx_chunk = ct[start:end]
    target.send_cmd(0x07, i, tx_chunk)
    ack = target.simpleserial_wait_ack()
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ERROR sending chunk {i}: ack={ack}")
    print(f"Sent chunk {i}/{num_chunks-1}: {len(tx_chunk)} bytes")
    time.sleep(0.02)

In [ ]:
# 3) Leer cada chunk: primero paquete 'k', luego ACK del comando 0x08
# 3) Read each chunk: 1st from the paket 'k', then the ACK from the comand 0x08
rec_ct = bytearray()
target.con()
for i in range(num_chunks):
    start = i * chunk_size
    expected_len = min(chunk_size, CT_LEN - start)

    target.send_cmd(0x08, i, [])

    # Leer respuesta de datos sin consumir ACK automaticamente
    rx_chunk = target.simpleserial_read('c', expected_len, ack=False)
    if rx_chunk is None:
        raise RuntimeError(f"Timeout reading chunk {i}")

    ack = target.simpleserial_wait_ack()
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ERROR ACK reading chunk {i}: ack={ack}")

    rec_ct.extend(rx_chunk)
    print(f"Receiving chunk {i}/{num_chunks-1}: {len(rx_chunk)} bytes")
    time.sleep(0.02)

In [ ]:
print("\n--- Verify ---")
if len(rec_ct) != len(ct):
    print(f"ERROR: Length mismatch. Expected: {len(ct)}, Received: {len(rec_ct)}")
else:
    if rec_ct == ct:
        print("Success: Received key same as the Sended ")
    else:
        print("ERROR: Key misstach ")
        # Encontrar primera diferencia
        # Find the first difference
        for i in range(len(ct)):
            if ct[i] != rec_ct[i]:
                print(f"First difference in byte {i}: expected 0x{ct[i]:02x}, received 0x{rec_ct[i]:02x}")
                break

In [ ]:
# SS_LEN = 32  test
# The chunk size can be up to 100 bytes in SAM4S
# For STM32 seze upto 240 bytes,
# Also the size can be go from 1-240 bytes without any issues
# But in some cases it's just a size adjustment.
SS_LEN = 32
chunk_size = 32  #Keep <= 100 to reduce the risk of UART overrun on SAM4S
num_chunks = (SS_LEN + chunk_size - 1) // chunk_size

# Limpiar basura de ejecuciones anteriores para evitar desincronizacion del parser
# Clean the trash from previous executions and avoid parser desynchronization
target.reset_comms()
target.flush()

# Verificar comandos disponibles en el firmware cargado
# Verify vailable commands in the loaded firmware
cmds = target.get_simpleserial_commands()
print("Reported comands:", cmds)

# 1) Configurar chunk size en firmware y esperar ACK
# 1) Configure chunk size in the firmware and wait ACK
target.send_cmd(0x04, 0, [chunk_size & 0xFF, (chunk_size >> 8) & 0xFF])
ack = target.simpleserial_wait_ack()
print(f"ACK 0x04: {ack}")
if ack is None or len(ack) == 0 or ack[0] != 0x00:
    raise RuntimeError(f"Fallo en cmd 0x04, ack={ack}")
time.sleep(0.05)

In [ ]:
# 2) Enviar Pk en chunks con ACK por cada envio
# 2) Send the secret key using chunks.
for i in range(num_chunks):
    start = i * chunk_size
    end = min(start + chunk_size, SS_LEN)
    tx_chunk = ss[start:end]
    target.send_cmd(0x09, i, tx_chunk)
    ack = target.simpleserial_wait_ack()
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ERROR sending chunk {i}: ack={ack}")
    print(f"Sent chunk {i}/{num_chunks-1}: {len(tx_chunk)} bytes")
    time.sleep(0.02)

In [ ]:
# 3) Leer cada chunk: primero paquete 'p', luego ACK del comando 0x0A
# 3) Read each chunk: 1st from the paket 'p', then the ACK from the comand 0x0A
rec_ss = bytearray()
for i in range(num_chunks):
    start = i * chunk_size
    expected_len = min(chunk_size, SS_LEN - start)

    target.send_cmd(0x0A, i, [])

    # Leer respuesta de datos sin consumir ACK automaticamente
    # Read response from data without automaic ACK consumption
    rx_chunk = target.simpleserial_read('s', expected_len, ack=False)
    if rx_chunk is None:
        RuntimeError(f"Timeout reading chunk {i}")

    ack = target.simpleserial_wait_ack()
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ERROR ACK reading chunk {i}: ack={ack}")

    rec_ss.extend(rx_chunk)
    print(f"Receiving chunk {i}/{num_chunks-1}: {len(rx_chunk)} bytes")
    time.sleep(0.02)

In [ ]:
print("\n--- Verify ---")
if len(rec_ss) != len(ss):
    print(f"ERROR: Length mismatch. Expected: {len(pk)}, Received: {len(rec_pk)}")
else:
    if rec_ss == ss:
        print("Success: Received key same as the Sended ")
    else:
        print("ERROR: Key misstach ")
        # Encontrar primera diferencia
        # Find the first difference
        for i in range(len(ss)):
            if ss[i] != rec_ss[i]:
                print(f"First difference in byte {i}: expected 0x{ss[i]:02x}, received 0x{rec_ss[i]:02x}")
                break

In [13]:
# LED control via SimpleSerial2 for the current firmware.
# Command mapping:
#   'h' -> turn LED on
#   'l' -> turn LED off
#   't' -> toggle LED
# Payload is one byte with the LED GPIO ID.

# LED1 = gpio(16) for SAM4S
# LED2 = gpio(15) for SAM4S
# LED3 = gpio(14) for SAM4

# Remember to chek it out for each target not always are the same

LED1 = 16 # 16 GPIO pin number
LED2 = 15 # 15 GPIO pin number
LED3 = 14 # 14 GPIO pin number

def led_on(led_id: int):
    # Send "on" command and wait for ACK from target
    target.send_cmd(ord("h"), 0x00, bytearray([led_id]))
    return target.simpleserial_wait_ack()

def led_off(led_id: int):
    # Send "off" command and wait for ACK from target
    target.send_cmd(ord("l"), 0x00, bytearray([led_id]))
    return target.simpleserial_wait_ack()

def led_toggle(led_id: int):
    # Send "toggle" command and wait for ACK from target
    target.send_cmd(ord("t"), 0x00, bytearray([led_id]))
    return target.simpleserial_wait_ack()

# Example usage (LED1):
print("LED1 ON     ->", led_on(LED1))
#print("LED1 OFF    ->", led_off(LED1))
#print("LED1 TOGGLE ->", led_toggle(LED1))

# You can also use LED2 / LED3:
# print("LED2 ON     ->", led_on(LED2))
# print("LED2 OFF    ->", led_off(LED2))
# print("LED2 TOGGLE ->", led_toggle(LED2))
#
# print("LED3 ON     ->", led_on(LED3))
# print("LED3 OFF    ->", led_off(LED3))
# print("LED3 TOGGLE ->", led_toggle(LED3))

LED1 ON     -> bytearray(b'\x00')


In [14]:
# Minimal raw command version (without helper functions)

target.send_cmd(ord("h"), 0x00, bytearray([16]))  # LED1 ON
print(target.simpleserial_wait_ack())

target.send_cmd(ord("l"), 0x00, bytearray([16]))  # LED1 OFF
print(target.simpleserial_wait_ack())

target.send_cmd(ord("t"), 0x00, bytearray([16]))  # LED1 TOGGLE
print(target.simpleserial_wait_ack())

bytearray(b'\x00')
bytearray(b'\x00')
bytearray(b'\x00')


In [15]:
import time

LED1 = 16
LED2 = 15
LED3 = 14

def wait_ack_ok(target, tries=6, timeout=200):
    for _ in range(tries):
        ack = target.simpleserial_wait_ack(timeout=timeout)
        if ack is None:
            continue
        v = ack[0] if isinstance(ack, (bytes, bytearray)) else int(ack)
        if v == 0x00:
            return True
        # 0x01 puede ser comando inválido real o paquete viejo; seguimos drenando
    return False

def led_cmd(cmd_char, led_id):
    target.reset_comms()
    target.flush()
    target.send_cmd(ord(cmd_char), 0x00, bytearray([led_id]))
    ok = wait_ack_ok(target)
    print(f"{cmd_char} LED={led_id} -> {'OK' if ok else 'FAIL'}")
    return ok

h LED=16 -> OK
l LED=16 -> OK
t LED=16 -> OK


True

In [22]:
led_cmd('h', LED1)   # ON
time.sleep(0.2)
led_cmd('l', LED1)   # OFF
time.sleep(0.2)
led_cmd('t', LED1)   # TOGGLE

h LED=16 -> OK
l LED=16 -> OK
t LED=16 -> OK


True

In [23]:
led_cmd('h', LED2)   # ON
time.sleep(0.2)
led_cmd('l', LED2)   # OFF
time.sleep(0.2)
led_cmd('t', LED2)   # TOGGLE

h LED=15 -> OK
l LED=15 -> OK
t LED=15 -> OK


True

In [24]:
led_cmd('h', LED3)   # ON
time.sleep(0.2)
led_cmd('l', LED3)   # OFF
time.sleep(0.2)
led_cmd('t', LED3)   # TOGGLE

h LED=14 -> OK
l LED=14 -> OK
t LED=14 -> OK


True

A simple `target.read()` will return all the characters that have been sent back from the target so far. Let's see what the device returned to us:

In [ ]:
recv_msg += target.read() #you might have to run this block a few times to get the full message
print(recv_msg)

The simpleserial commands are usually sufficient for taking to simpleserial firmware, but you'll need the raw serial commands for some of the other labs.

## SimpleSerial 2

As of ChipWhisperer 5.4, a new target communication protocol, SimpleSerial 2 is available. It has a number of advantages over the old SimpleSerial protocol:

1. Data is unencoded binary (except for byte stuffing) instead of ACII encoded, meaning far more data can be sent per packet (just under double)
1. It's got a command and subcommand field that get passed to the target callback functions, meaning more can be done per packet. For example, the simpleserial-aes firmware has been modified to allow plaintext, key, and/or mask to be set with a single packet.
1. It's got a length field, meaning the same target commands can take different length packets depending on the situation. This, for example, allows both plaintext and key (or only one) to be send with the same command in simpleserial-aes.
1. It's got an 8-bit CRC (0xA6) for data integrety.
1. Frames are consistant overhead byte stuffed (COBS). This makes it easy to reset communication (done simply by sending two 0x00 bytes) and helps catch malformed length bytes.

It is, however, a more complicated protocol. You can use it by compiling firmware with `SS_VER=SS_VER_2_0` and using `cw.targets.SimpleSerial2`. In `sca101` and `fault101`, it can be used by setting `SS_VER='SS_VER_2_0'` when setting `PLATFORM` and `SCOPETYPE`.

## Capturing Traces

Now that the target's programmed and we know how to communicate with it, let's start recording some power traces! To capture a trace:

1. Arm the ChipWhisperer with `scope.arm()`. It will begin capturing as soon as it is triggered (which in our case is a rising edge on `gpio4`.
1. `scope.capture()` will read back the captured power trace, blocking until either ChipWhisperer is done recording, or the scope times out. Note that the error return will tell you whether or not the scope timed out. It does not return the captured scope data.
1. You can read back the captured power trace with `scope.get_last_trace()`.

`simpleserial_base` will trigger the ChipWhisperer when we send the `'p'` command. Try capturing a trace now:

ChipWhisperer also has a `capture_trace()` convience function that:

1. Optionally sends the `'k'` command
1. Arms the scope
1. Sends the `'p'` command
1. Captures the trace
1. Reads the return `'r'` message
1. Returns a `Trace` class that groups the trace data, `'p'` message, the `'r'` message, and the `'k'` message

It isn't always the best option to use, but it's usually sufficient for most simpleserial applications

## Conclusion

And that's it! You should be all ready to continue on to SCA101 (or any other of our courses).

We've glossed over some stuff here, so consult the [API documentation](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html) or ask on our [forums](https://forum.newae.com) if you get stuck.

As a final step, we should disconnect from the hardware so it doesn't stay "in use" by this notebook.

In [ ]:
scope.dis()
target.dis()

## ChipWhisperer-Husky

While the ChipWhisperer-Husky has many new features, its API has been designed to keep most basic tasks the same, especially with power analysis. You should be able to go through SCA101 without learning anything Husky specific and fault101 will cover any Husky specific things you need to know. That being said, you may still find it helpful to refer to the `demos/husky` folder, starting with [01 - Introduction to ChipWhisperer-Husky.ipynb](demos/husky/01%20-%20Introduction%20to%20ChipWhisperer-Husky.ipynb) to learn more about its advanced features.